In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded")

Libraries loaded


In [2]:
transaction = pd.read_csv('../train_transaction.csv')
identity = pd.read_csv('../train_identity.csv')

print("Transaction shape:", transaction.shape)
print("Identity shape:", identity.shape)

Transaction shape: (590540, 394)
Identity shape: (144233, 41)


In [3]:
print("Fraud distribution:")
print(transaction['isFraud'].value_counts())
print("\nFraud rate:", round(transaction['isFraud'].mean() * 100, 2), "%")

Fraud distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Fraud rate: 3.5 %


In [4]:
print("Transaction columns:")
print(transaction.columns.tolist())

Transaction columns:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V

In [5]:
print("Identity columns:")
print(identity.columns.tolist())

Identity columns:
['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [6]:
trans_nulls = (transaction.isnull().sum() / len(transaction) * 100).round(1)
id_nulls = (identity.isnull().sum() / len(identity) * 100).round(1)

print("Transaction columns with >50% nulls:")
print(trans_nulls[trans_nulls > 50].sort_values(ascending=False))

print("\nIdentity columns with >50% nulls:")
print(id_nulls[id_nulls > 50].sort_values(ascending=False))

Transaction columns with >50% nulls:
dist2    93.6
D7       93.4
D14      89.5
D13      89.5
D12      89.0
         ... 
M5       59.3
M9       58.6
M8       58.6
M7       58.6
D5       52.5
Length: 174, dtype: float64

Identity columns with >50% nulls:
id_24    96.7
id_07    96.4
id_08    96.4
id_21    96.4
id_22    96.4
id_23    96.4
id_25    96.4
id_26    96.4
id_27    96.4
id_18    68.7
id_03    54.0
id_04    54.0
dtype: float64


In [7]:
key_cols = ['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 
            'card5', 'card6', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain']

print("Null % in key columns:")
print((transaction[key_cols].isnull().sum() / len(transaction) * 100).round(1))

Null % in key columns:
TransactionAmt     0.0
ProductCD          0.0
card1              0.0
card2              1.5
card3              0.3
card4              0.3
card5              0.7
card6              0.3
addr1             11.1
addr2             11.1
P_emaildomain     16.0
R_emaildomain     76.8
dtype: float64


## Data Quality Notes

**Key columns are largely complete** — TransactionAmt, ProductCD, and card1 have zero nulls, 
giving us a solid foundation for the tabular model.

**Email domains have significant missingness** — P_emaildomain (16% missing) will be used in 
the NLP module with a fallback category for nulls. R_emaildomain (76.8% missing) will be used 
only as a supporting signal where available.

**174 out of 394 transaction columns have >50% nulls** — mostly V-series (Vesta engineered) 
and D-series (time delta) features. We do not drop these blindly; some sparse V columns carry 
strong fraud signal and will be evaluated during feature selection in notebook 02.

**Identity table is sparse by design** — only 144k of 590k transactions have identity records. 
The join will be a left join, preserving all transactions and leaving identity fields null where 
unavailable.

In [8]:
print("ProductCD value counts:")
print(transaction['ProductCD'].value_counts())

print("\nTop 20 P_emaildomain values:")
print(transaction['P_emaildomain'].value_counts().head(20))

print("\nTop 10 card4 values:")
print(transaction['card4'].value_counts())

print("\nTop 10 card6 values:")
print(transaction['card6'].value_counts())

ProductCD value counts:
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

Top 20 P_emaildomain values:
P_emaildomain
gmail.com        228355
yahoo.com        100934
hotmail.com       45250
anonymous.com     36998
aol.com           28289
comcast.net        7888
icloud.com         6267
outlook.com        5096
msn.com            4092
att.net            4033
live.com           3041
sbcglobal.net      2970
verizon.net        2705
ymail.com          2396
bellsouth.net      1909
yahoo.com.mx       1543
me.com             1522
cox.net            1393
optonline.net      1011
charter.net         816
Name: count, dtype: int64

Top 10 card4 values:
card4
visa                384767
mastercard          189217
american express      8328
discover              6651
Name: count, dtype: int64

Top 10 card6 values:
card6
debit              439938
credit             148986
debit or credit        30
charge card            15
Name: count, dtype: int64


## NLP Module Preview — Email Domain Signal

The `P_emaildomain` column is the primary text feature for Module 2. Key observations:

- **anonymous.com** is the 4th most common domain (~37k transactions) — this is not a real 
  email provider. Transactions using this domain are likely masking identity, a known fraud pattern.
- Legitimate providers (gmail, yahoo, hotmail) dominate but fraud rate varies significantly 
  across them — gmail fraud rate differs from anonymous.com fraud rate.
- Domain-level features we will engineer: domain type (free/corporate/anonymous), 
  TLD extraction, fraud rate encoding per domain.

`R_emaildomain` (recipient) is 76.8% null but where present, mismatch between 
P and R domains is a strong fraud signal.

In [9]:
fraud_by_email = transaction.groupby('P_emaildomain')['isFraud'].agg(['mean', 'count'])
fraud_by_email.columns = ['fraud_rate', 'count']
fraud_by_email = fraud_by_email[fraud_by_email['count'] > 500]
fraud_by_email = fraud_by_email.sort_values('fraud_rate', ascending=False)
print(fraud_by_email.head(15))

               fraud_rate   count
P_emaildomain                    
mail.com         0.189624     559
outlook.com      0.094584    5096
live.com.mx      0.054740     749
hotmail.com      0.052950   45250
gmail.com        0.043542  228355
icloud.com       0.031434    6267
comcast.net      0.031187    7888
charter.net      0.030637     816
bellsouth.net    0.027763    1909
live.com         0.027622    3041
anonymous.com    0.023217   36998
yahoo.com        0.022757  100934
msn.com          0.021994    4092
aol.com          0.021811   28289
earthlink.net    0.021401     514


In [10]:
print("DeviceType value counts:")
print(identity['DeviceType'].value_counts())

print("\nTop 15 DeviceInfo values:")
print(identity['DeviceInfo'].value_counts().head(15))

DeviceType value counts:
DeviceType
desktop    85165
mobile     55645
Name: count, dtype: int64

Top 15 DeviceInfo values:
DeviceInfo
Windows                        47722
iOS Device                     19782
MacOS                          12573
Trident/7.0                     7440
rv:11.0                         1901
rv:57.0                          962
SM-J700M Build/MMB29K            549
SM-G610M Build/MMB29K            461
SM-G531H Build/LMY48B            410
rv:59.0                          362
SM-G935F Build/NRD90M            334
SM-G955U Build/NRD90M            328
SM-G532M Build/MMB29T            316
ALE-L23 Build/HuaweiALE-L23      312
SM-G950U Build/NRD90M            290
Name: count, dtype: int64


In [11]:
df = transaction.merge(identity, on='TransactionID', how='left')

print("Merged shape:", df.shape)
print("Fraud rate after merge:", round(df['isFraud'].mean() * 100, 2), "%")

Merged shape: (590540, 434)
Fraud rate after merge: 3.5 %


## Data Loading Complete

- **Transaction table:** 590,540 rows × 394 columns
- **Identity table:** 144,233 rows × 41 columns  
- **Merged dataset:** 590,540 rows × 434 columns (left join on TransactionID)
- **Fraud rate:** 3.5% — significant class imbalance to be handled in notebook 02 via class weights
- Only 144k of 590k transactions have identity data — identity columns will be null for the rest, which is expected and handled downstream

Column groups for reference:
- `TransactionAmt`, `ProductCD` — core transaction features
- `card1–card6` — payment card attributes
- `P_emaildomain`, `R_emaildomain` — NLP module (notebook 03)
- `C1–C14` — count features (linked cards, addresses per account)
- `D1–D15` — time delta features
- `M1–M9` — match features (name/address verification flags)
- `V1–V339` — Vesta engineered features, evaluated in notebook 02
- `DeviceType`, `DeviceInfo` — NLP module (notebook 03)
- `id_01–id_38` — anonymized identity features

In [14]:
df.to_csv('../outputs/merged_data.csv', index=False)
print("Saved to outputs/merged_data.csv")

Saved to outputs/merged_data.csv
